In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 4 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# - maximise raw objective
# - use all cumulative observations through Week 7
# - fit ARD Matern GP
# - optimise GP hyperparameters automatically
# - generate local + wide + global candidates
# - EI is primary acquisition
# - UCB is used to inspect exploration/exploitation balance

In [2]:
X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 4

print("X shape:", X.shape)
print("Y shape:", Y.shape)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (37, 4)
Y shape: (37,)

Current best observed input:
[0.357812 0.420904 0.424244 0.430762]

Current best observed output:
0.601059871344695

Y range:
min = -32.625660215962455
max = 0.601059871344695
std = 9.264832453090829


In [3]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
2.34**2 * Matern(length_scale=[1.61, 1.24, 1.25, 1.34], nu=2.5) + WhiteKernel(noise_level=0.000457)


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[1.61097814 1.24286102 1.25255697 1.34465321]

Normalised inverse-lengthscale sensitivity:
[0.20918756 0.27114583 0.26904691 0.2506197 ]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2]


In [6]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(60000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(30000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(60000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print("Candidates before filtering:", len(candidates))

Candidates before filtering: 150000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.008
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 149998


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Predictions complete.")

Predictions complete.


In [9]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [0.32109407 0.45511538 0.44684827 0.43055742]
mean = 0.37805899100467677
std = 0.31842890652708605
EI = 0.04547257607093111


In [11]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [0.32109407 0.45511538 0.44684827 0.43055742] 
 mean = 0.378059 
 std = 0.318429 
 EI = 0.04547258 

xi = 9.264832e-02 
 candidate = [0.32109407 0.45511538 0.44684827 0.43055742] 
 mean = 0.378059 
 std = 0.318429 
 EI = 0.02697401 

xi = 4.632416e-01 
 candidate = [0.26773554 0.4655458  0.45065301 0.4487938 ] 
 mean = 0.028561 
 std = 0.487384 
 EI = 0.00294062 

xi = 9.264832e-01 
 candidate = [0.16921119 0.46856238 0.4840021  0.49001734] 
 mean = -1.566026 
 std = 1.020798 
 EI = 0.00035006 



In [12]:
print("\nUCB diagnostic:\n")

for beta in [
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [0.33509331 0.4385294  0.42560279 0.42134542] 
 mean = 0.448359 
 std = 0.249177 
 UCB = 0.473277 

beta=0.25 
 candidate = [0.33509331 0.4385294  0.42560279 0.42134542] 
 mean = 0.448359 
 std = 0.249177 
 UCB = 0.510653 

beta=0.5 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.443847 
 std = 0.260506 
 UCB = 0.574099 

beta=1.0 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.443847 
 std = 0.260506 
 UCB = 0.704352 



In [13]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.33509331 0.4385294  0.42560279 0.42134542]
mean = 0.44835904691155015
std = 0.24917732593548475


In [14]:
# --------------------------------------------------
# Final Function 4 Week 8 selection
# --------------------------------------------------
#
# EI, UCB and highest GP mean all identify the same
# general region around the incumbent.
#
# Increasing beta produces a gradual move toward
# slightly greater uncertainty rather than an
# unstable boundary solution.
#
# beta = 0.5 is selected to provide a balanced
# exploration-exploitation trade-off.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week8_candidate = candidates[final_idx]

print("Week 8 Function 4 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 4 candidate:
[0.33524044 0.4414815  0.43811063 0.42398236]

Predicted mean:
0.44384663985885453

Predicted std:
0.26050571320058924

UCB:
0.5740994964591492

Portal format:
0.335240-0.441481-0.438111-0.423982
